# Phase 4 - Experiment 3: knowledge distillation into the D encoder (mandatory KD experiment)

**Teacher:** experiment 1 = A2 full fine-tuning (70.1M parameters, 10 causal layers, reads the last 1536 tokens,
dev macro-F1 0.810). Its logits on all 31,652 training cases are already saved (`train_logits.npy`).

**Student:** the **4-layer bidirectional encoder of my Phase-1 encoder-decoder D** (~42M parameters, 40% smaller,
4 instead of 10 layers) -> attention pooling -> Linear(640 -> 2). It reads the **last 1024 tokens**
(D's encoder only ever saw inputs of <= 1024 tokens in Phase 1). All student weights are trained.

**KD loss** (Hinton et al. 2015), per case:
`loss = (1 - a) * CE(student, gold label) + a * T^2 * KL( softmax(teacher / T) || softmax(student / T) )`, a = 0.5, T = 2.
- CE: learn the gold outcome (class-weighted, as in all experiments).
- KL: imitate the teacher's probabilities. T = 2 softens them, so a case where the teacher says 0.55 teaches
  "this one is uncertain", not just "label 1". T^2 keeps both parts of the loss at a similar size.
- The teacher was trained on the same training cases, so its soft labels are a little over-confident; still, only
  38% of them are above 0.95 (mean top probability 0.85), so they carry information.

**Control (dev only, not a submission):** the same student trained **without** the teacher -> shows what KD adds.

Report: dev macro-F1, parameters, inference speed and memory of teacher vs student vs control.
Run folders: `checkpoints/phase4/D_full_bidir_last1024_attn_kd2/` and `.../D_full_bidir_last1024_attn/`
(if `scripts/p4_run.py kd` already ran them, every cell just loads).

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, cjpe_utils, cjpe_models, cjpe_experiment
for m in (cjpe_utils, cjpe_models, cjpe_experiment):
    importlib.reload(m)
from cjpe_experiment import Experiment, compare
import cjpe_models as cm

TEACHER = "A2_full_causal_last1536_attn"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU would be far too slow)")

## 1. Teacher soft labels: how confident is the teacher on the training cases?

In [ ]:
t_logits = np.load(ROOT / "checkpoints" / "phase4" / TEACHER / "train_logits.npy")
for T in (1, 2, 4):
    p = torch.softmax(torch.from_numpy(t_logits) / T, -1).numpy()
    top = p.max(1)
    print(f"T={T}: mean top probability {top.mean():.3f} | cases with top > 0.95: {100 * (top > 0.95).mean():.1f}%")

## 2. Student WITH knowledge distillation

In [ ]:
kd = Experiment(ROOT, {"phase1": "D", "method": "full", "kd_teacher": TEACHER})
kd.memory_check()
_ = kd.train()

In [ ]:
rep_kd = kd.report()
zip_kd = kd.predict_test()
kd.speed(); kd.log()

## 3. Control: same student WITHOUT the teacher (dev only)

In [ ]:
ctrl = Experiment(ROOT, {"phase1": "D", "method": "full"})
_ = ctrl.train()
rep_ctrl = ctrl.report()
_ = ctrl.predict_test()     # written, but not planned for upload
ctrl.speed(); ctrl.log()

## 4. Teacher vs student vs control

Speeds are measured on the same 200 dev cases (teacher reads 1536 tokens per case, students 1024).

In [ ]:
rows = {}
for name, label in [(TEACHER, "teacher: A2 full FT"), (kd.name, "student: D + KD"), (ctrl.name, "control: D, no KD")]:
    d = ROOT / "checkpoints" / "phase4" / name
    rep, sp = json.load(open(d / "dev_report.json")), json.load(open(d / "speed.json"))
    rows[label] = {"params_M": sp["params_M"], "dev F1 @0.5": rep["at_0.5"]["macro_f1"], "dev F1 tuned": rep["tuned"]["macro_f1"],
                   "threshold": rep["threshold"], "cases/s": sp["cases_per_s"], "tokens/s": sp["tokens_per_s"], "GB": sp["peak_gb"]}
tab = pd.DataFrame(rows).T
display(tab)
t, s, c = tab.iloc[0], tab.iloc[1], tab.iloc[2]
print(f"student keeps {100 * s['dev F1 tuned'] / t['dev F1 tuned']:.1f}% of the teacher's dev macro-F1 with "
      f"{100 * s['params_M'] / t['params_M']:.0f}% of the parameters, {s['cases/s'] / t['cases/s']:.2f}x the cases/s")
print(f"KD effect (student - control): {s['dev F1 tuned'] - c['dev F1 tuned']:+.4f} dev macro-F1 (tuned)")

# agreement with the teacher on dev: does KD make the student behave more like the teacher?
pt = cm.probs_1(np.load(ROOT / "checkpoints" / "phase4" / TEACHER / "dev_logits.npy"))
for label, name, thr in [("student + KD", kd.name, s["threshold"]), ("control", ctrl.name, c["threshold"])]:
    ps = cm.probs_1(np.load(ROOT / "checkpoints" / "phase4" / name / "dev_logits.npy"))
    agree = ((ps >= thr) == (pt >= t["threshold"])).mean()
    print(f"{label}: agrees with the teacher on {100 * agree:.1f}% of dev cases, correlation of P(1) {np.corrcoef(ps, pt)[0, 1]:.3f}")
compare(ROOT)